In [ ]:
"""
Fashion House Classification - Custom CNN + ResNet50 Transfer Learning
------------------------------------------------------------------------
Classifies runway fashion images into 15 fashion houses (Chanel, Gucci,
Dior, etc.) using a custom CNN built from scratch and ResNet50 transfer
learning (feature-extraction and fine-tuning modes), with MixUp
augmentation, class-imbalance handling, Optuna hyperparameter tuning,
Grad-CAM interpretability, a soft-voting ensemble, and a Gradio demo.

Dataset: Vogue Runway (HuggingFace: tonyassi/vogue-runway-top15-512px),
filtered to 2015-2019 collections (~22,700 images, 15 classes).
Result: ResNet50 (fine-tuned) reaches ~95% validation accuracy.

Requirements:
    pip install datasets transformers timm optuna gradio torchcam \
        scikit-learn matplotlib seaborn torchvision tqdm torch
"""

In [ ]:
import os, warnings, random
import numpy as np
import pandas as pd
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
from collections import Counter
import random
from tqdm.auto import tqdm
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms as T
import torchvision.models as models
import timm

from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score
)
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# Device Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'  Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Constraints
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 20
LR          = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 2

print('\n All imports successful!')

In [ ]:
# Data understanding
# Loading the dataset
from datasets import load_dataset

print('Loading Vogue Runway dataset...')
ds = load_dataset('tonyassi/vogue-runway-top15-512px')
print(ds)

# Inspect structure
print('\n Dataset features:', ds['train'].features)
print(f'\n Train samples : {len(ds["train"])}')
print(f' Available splits: {list(ds.keys())}')

In [ ]:
# Class distribution analysis: KEEPING EVERY HOUSE ONLY BETWEEN THE YEARS OF 2015 TO 2019 FOR A NARROWER DISTRIBUTION.
# Filter to only keep 2015-2019 collections
KEEP_YEARS = ['2015', '2016', '2017', '2018', '2019']

# Get original label names FIRST before using them
label_names = ds['train'].features['label'].names

def is_valid_label(example):
    label_name = label_names[example['label']]
    return any(year in label_name for year in KEEP_YEARS)

ds_filtered = ds['train'].filter(is_valid_label)

# Merge calvin klein collection → calvin klein
def get_brand(label_id):
    brand = label_names[label_id].split(',')[0].strip()
    if brand == 'calvin klein collection':
        brand = 'calvin klein'
    return brand

# Remap labels to brand-only (15 classes)
unique_brands = sorted(set(get_brand(l) for l in ds_filtered['label']))
brand_to_idx  = {brand: idx for idx, brand in enumerate(unique_brands)}

def remap_label(example):
    example['label'] = brand_to_idx[get_brand(example['label'])]
    return example

ds_filtered = ds_filtered.map(remap_label)

# Now use remapped dataset
train_labels = ds_filtered['label']
NUM_CLASSES  = len(unique_brands)

print(f'  Number of fashion houses: {NUM_CLASSES}')
print(f'  Original dataset size   : {len(ds["train"])}')
print(f'  Filtered dataset size   : {len(ds_filtered)}')

print('\nFashion Houses:')
for i, name in enumerate(unique_brands):
    print(f'  [{i:2d}] {name}')

# Count per class
label_counts  = Counter(train_labels)
counts_sorted = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)

In [1]:
# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Dataset Class Distribution Analysis (2015-2019)', fontsize=16, fontweight='bold')

# Bar chart
ax1 = axes[0]
names  = [unique_brands[i] for i, _ in counts_sorted]
counts = [c for _, c in counts_sorted]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, NUM_CLASSES))
bars = ax1.barh(names, counts, color=colors)
ax1.set_xlabel('Number of Images', fontsize=12)
ax1.set_title('Images per Fashion House (2015-2019)', fontsize=13)
for bar, count in zip(bars, counts):
    ax1.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             str(count), va='center', fontsize=9)
ax1.invert_yaxis()

# Pie chart
ax2 = axes[1]
ax2.pie(counts, labels=names, autopct='%1.1f%%',
        colors=colors, startangle=140,
        textprops={'fontsize': 8})
ax2.set_title('Class Distribution (%)', fontsize=13)

plt.tight_layout()
plt.show()

# Imbalance ratio pie
ax2 = axes[1]
ax2.pie(counts, labels=names, autopct='%1.1f%%',
        colors=colors, startangle=140,
        textprops={'fontsize': 7})
ax2.set_title('Class Share Distribution (2015-2019)', fontsize=13)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Imbalance metric
max_c, min_c = max(counts), min(counts)
max_brand = names[counts.index(max_c)]
min_brand = names[counts.index(min_c)]
print(f'\n Class Imbalance Ratio (max/min): {max_c/min_c:.2f}x')
print(f'   Most images  : {max_brand} ({max_c} images)')
print(f'   Least images : {min_brand} ({min_c} images)')
print('   → Will use WeightedRandomSampler to compensate')

# Visualise sample images per class
import math
from PIL import Image

# Group indices by label using filtered dataset
label_to_indices = {i: [] for i in range(NUM_CLASSES)}
for idx, lbl in enumerate(ds_filtered['label']):
    label_to_indices[lbl].append(idx)

n_show = 5  # samples per class
cols = n_show
rows = NUM_CLASSES

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
fig.suptitle('Sample Images per Fashion House (2015-2019)', fontsize=18,
             fontweight='bold', y=1.01)

for class_id in range(NUM_CLASSES):
    sample_ids = random.sample(label_to_indices[class_id],
                               min(n_show, len(label_to_indices[class_id])))
    for col, idx in enumerate(sample_ids):
        img = ds_filtered[idx]['image']
        ax  = axes[class_id][col]
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(unique_brands[class_id], fontsize=9,
                          rotation=0, labelpad=60, va='center',
                          fontweight='bold')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Sample images visualized.')

NameError: name 'plt' is not defined

In [ ]:
# ── Setup ─────────────────────────────────────────────────────
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224

# Custom CNN architecture from scratch
class FashionCNNScratch(nn.Module):
    """
    Custom CNN built from first principles for 15 Fashion House Classification.
    Trained on Vogue Runway dataset filtered to 2015-2019 collections.
    Architecture follows the VGG-style philosophy:
    stack of 3x3 convolutions with progressive depth doubling.

    Feature hierarchy:
      Block 1 (32 filters)  → low-level: edges, gradients
      Block 2 (64 filters)  → mid-level: textures, fabric patterns
      Block 3 (128 filters) → high-level: garment parts
      Block 4 (256 filters) → semantic:  brand-specific features
      Block 5 (256 filters) → semantic:  fine-grained brand features
    """
    def __init__(self, num_classes: int = NUM_CLASSES, dropout: float = 0.5):
        super().__init__()

        self.block1 = self._make_block(3,   32)
        self.block2 = self._make_block(32,  64)
        self.block3 = self._make_block(64,  128)
        self.block4 = self._make_block(128, 256)
        self.block5 = self._make_block(256, 256)

        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout * 0.6),
            nn.Linear(256, num_classes)
        )

        self._init_weights()

    def _make_block(self, in_ch: int, out_ch: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.global_avg_pool(x)
        x = self.classifier(x)
        return x

    def get_feature_maps(self, x: torch.Tensor):
        maps = {}
        x = self.block1(x); maps['block1'] = x.clone().detach()
        x = self.block2(x); maps['block2'] = x.clone().detach()
        x = self.block3(x); maps['block3'] = x.clone().detach()
        x = self.block4(x); maps['block4'] = x.clone().detach()
        return maps


# Instantiate model
model = FashionCNNScratch(num_classes=NUM_CLASSES).to(DEVICE)
print(f'Model output classes : {NUM_CLASSES}')
print(f'Brands               : {unique_brands}')
print(f'Device               : {DEVICE}')

# ── Sanity Check ──────────────────────────────────────────────
cnn_scratch = FashionCNNScratch(num_classes=NUM_CLASSES).to(DEVICE)
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out   = cnn_scratch(dummy)
print(f'✅ FashionCNNScratch output shape: {out.shape}')  # (2, 15)

# Parameter count
total_params = sum(p.numel() for p in cnn_scratch.parameters())
trainable    = sum(p.numel() for p in cnn_scratch.parameters() if p.requires_grad)
print(f'   Total params    : {total_params:,}')
print(f'   Trainable params: {trainable:,}')

# Architecture Diagram (Text-based)
print("""
╔══════════════════════════════════════════════════════════════╗
║      FashionCNNScratch — Architecture Overview               ║
║      Dataset: Vogue Runway 2015-2019 | Classes: 15           ║
╠══════════════════════════════════════════════════════════════╣
║  Input: [B, 3, 224, 224]                                     ║
║      ↓                                                       ║
║  Block 1: Conv(3→32) → BN → ReLU → Conv → BN → ReLU         ║
║           MaxPool(2×2) → [B, 32, 112, 112]                   ║
║           ↑ LOW-LEVEL: edges, gradients, basic shapes        ║
║      ↓                                                       ║
║  Block 2: Conv(32→64) → BN → ReLU → Conv → BN → ReLU        ║
║           MaxPool(2×2) → [B, 64, 56, 56]                     ║
║           ↑ MID-LEVEL: fabric textures, weave patterns       ║
║      ↓                                                       ║
║  Block 3: Conv(64→128) → BN → ReLU → Conv → BN → ReLU       ║
║           MaxPool(2×2) → [B, 128, 28, 28]                    ║
║           ↑ HIGH-LEVEL: garment parts, lapels, collars       ║
║      ↓                                                       ║
║  Block 4: Conv(128→256) → BN → ReLU → Conv → BN → ReLU      ║
║           MaxPool(2×2) → [B, 256, 14, 14]                    ║
║           ↑ SEMANTIC: brand-defining silhouettes             ║
║      ↓                                                       ║
║  Block 5: Conv(256→256) → BN → ReLU → Conv → BN → ReLU      ║
║           MaxPool(2×2) → [B, 256, 7, 7]                      ║
║      ↓                                                       ║
║  GlobalAvgPool → [B, 256, 1, 1]                              ║
║  Flatten → [B, 256]                                          ║
║  FC(256→512) → BN → ReLU → Dropout(0.5)                     ║
║  FC(512→256) → BN → ReLU → Dropout(0.3)                     ║
║  FC(256→15)  → Logits (15 Fashion Houses)                    ║
╚══════════════════════════════════════════════════════════════╝
""")

In [ ]:
# feature map visualisation
def visualize_feature_maps(model, img_tensor, title='Feature Maps — Vogue Runway 2015-2019'):
    """Display first 8 channels of each conv block output for 15-class fashion model."""
    model.eval()
    with torch.no_grad():
        feat_maps = model.get_feature_maps(img_tensor.unsqueeze(0).to(DEVICE))

    fig, big_axes = plt.subplots(1, len(feat_maps),
                                  figsize=(5 * len(feat_maps), 6))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    for ax_idx, (block_name, fmap) in enumerate(feat_maps.items()):
        # fmap shape: [1, C, H, W]
        fmap = fmap[0].cpu().numpy()  # [C, H, W]
        n_show = min(8, fmap.shape[0])
        grid_size = int(math.ceil(n_show ** 0.5))

        inner_fig = plt.figure(figsize=(4, 4))
        for i in range(n_show):
            ax = inner_fig.add_subplot(grid_size, grid_size, i + 1)
            ax.imshow(fmap[i], cmap='viridis')
            ax.axis('off')
        inner_fig.suptitle(f'{block_name} | Classes: {NUM_CLASSES} brands', fontsize=10)
        inner_fig.savefig(f'feature_map_{block_name}.png',
                          dpi=100, bbox_inches='tight')
        plt.close(inner_fig)
        print(f'  Saved: feature_map_{block_name}.png  |  '
              f'map shape: {fmap.shape}')

print(f'visualize_feature_maps() defined for {NUM_CLASSES} classes: {unique_brands}')

In [ ]:
# ── Global Constants ──────────────────────────────────────────
SEED        = 42
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Device     : {DEVICE}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Seed       : {SEED}')

# custom dataset and transformation
class VogueRunwayDataset(Dataset):
    """Wrapper around HuggingFace dataset for PyTorch DataLoader.
       Filtered to 2015-2019 collections across 15 fashion houses."""

    def __init__(self, hf_dataset, transform=None):
        self.data      = hf_dataset
        self.transform = transform

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        item  = self.data[idx]
        image = item['image'].convert('RGB')  # ensure 3-channel
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# ── Transforms ────────────────────────────────────────────────
# ImageNet mean/std for pre-trained model normalisation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),   # slightly larger for random crop
    T.RandomCrop(IMG_SIZE),                      # random spatial crop
    T.RandomHorizontalFlip(p=0.5),               # mirror augmentation
    T.RandomRotation(degrees=10),                # slight rotation invariance
    T.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3,
        hue=0.1
    ),                                            # photometric augmentation
    T.RandomGrayscale(p=0.05),                   # occasional B&W
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.1, scale=(0.02, 0.1))    # CutOut: hides small patches
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# ── Train/Val split from filtered dataset ─────────────────────
# Since there is no test split, we split ds_filtered manually
from datasets import DatasetDict

split = ds_filtered.train_test_split(test_size=0.2, seed=SEED)
train_data = split['train']
val_data   = split['test']

In [ ]:
# ── Create Datasets ───────────────────────────────────────────
train_ds = VogueRunwayDataset(train_data, transform=train_transform)
val_ds   = VogueRunwayDataset(val_data,   transform=val_transform)

# ── Weighted Sampler for Class Imbalance ──────────────────────
# Assign higher sampling weight to under-represented classes
class_counts  = np.array([label_counts[i] for i in range(NUM_CLASSES)])
class_weights = 1.0 / class_counts
sample_weights = [class_weights[lbl] for lbl in train_data['label']]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f'DataLoaders ready.')
print(f'   Total filtered samples : {len(ds_filtered)}')
print(f'   Train samples          : {len(train_data)}')
print(f'   Val samples            : {len(val_data)}')
print(f'   Train batches          : {len(train_loader)}')
print(f'   Val   batches          : {len(val_loader)}')
print(f'   Classes                : {NUM_CLASSES} brands (2015-2019)')

# MixUp Augmentation (Advanced Regularisation)
def mixup_data(x: torch.Tensor, y: torch.Tensor,
               alpha: float = 0.2):
    """
    MixUp augmentation for 15-class Vogue Runway 2015-2019 dataset.
    Creates convex combinations of training pairs.
    Reference: Zhang et al., 2018 (https://arxiv.org/abs/1710.09412)

    lam ~ Beta(alpha, alpha)
    x_mix = lam * x_i + (1 - lam) * x_j
    The loss is then: lam * CE(y_i) + (1-lam) * CE(y_j)

    Why it works: Forces decision boundaries to be smooth,
    reducing overconfidence and improving calibration.
    Especially useful for our imbalanced 15-brand dataset.
    """
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print(f'MixUp augmentation defined for {NUM_CLASSES}-class dataset: {unique_brands}')

# Universal training engine
class EarlyStopping:
    """Monitor val_loss; restore best weights on plateau.
       Configured for 15-class Vogue Runway 2015-2019 dataset."""

    def __init__(self, patience: int = 5, delta: float = 1e-4,
                 path: str = 'best_model_2015_2019.pt'):
        self.patience  = patience
        self.delta     = delta
        self.path      = path
        self.counter   = 0
        self.best_loss = np.inf
        self.stop      = False

    def __call__(self, val_loss: float, model: nn.Module):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                print(f'   Early stopping triggered after {self.patience} epochs.')


class Trainer:
    """Full training/evaluation loop with AMP and Grad-Clip.
       Trains FashionCNNScratch on 15-class Vogue Runway 2015-2019 dataset."""

    def __init__(self, model, optimizer, scheduler, criterion,
                 device, model_name='FashionCNN_2015_2019', use_mixup=False):
        self.model      = model
        self.optimizer  = optimizer
        self.scheduler  = scheduler
        self.criterion  = criterion
        self.device     = device
        self.model_name = model_name
        self.use_mixup  = use_mixup
        self.scaler     = GradScaler()
        self.history    = {'train_loss': [], 'val_loss': [],
                           'train_acc': [],  'val_acc': []}

    def train_epoch(self, loader):
        self.model.train()
        total_loss = total_correct = total_samples = 0

        for imgs, labels in tqdm(loader, desc='  train', leave=False):
            imgs, labels = imgs.to(self.device), labels.to(self.device)

            if self.use_mixup:
                imgs, la, lb, lam = mixup_data(imgs, labels)

            self.optimizer.zero_grad(set_to_none=True)

            with autocast():
                logits = self.model(imgs)
                if self.use_mixup:
                    loss = mixup_criterion(self.criterion, logits, la, lb, lam)
                else:
                    loss = self.criterion(logits, labels)

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()

            preds = logits.argmax(dim=1)
            total_correct  += (preds == labels).sum().item()
            total_loss     += loss.item() * imgs.size(0)
            total_samples  += imgs.size(0)

        return total_loss / total_samples, total_correct / total_samples

    @torch.no_grad()
    def eval_epoch(self, loader):
        self.model.eval()
        total_loss = total_correct = total_samples = 0
        all_preds, all_labels = [], []

        for imgs, labels in tqdm(loader, desc='  eval ', leave=False):
            imgs, labels = imgs.to(self.device), labels.to(self.device)
            with autocast():
                logits = self.model(imgs)
                loss   = self.criterion(logits, labels)

            preds = logits.argmax(dim=1)
            total_correct  += (preds == labels).sum().item()
            total_loss     += loss.item() * imgs.size(0)
            total_samples  += imgs.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

        return (total_loss / total_samples,
                total_correct / total_samples,
                all_preds, all_labels)

    def fit(self, train_loader, val_loader, epochs: int,
            early_stopping: EarlyStopping = None):
        print(f'\nTraining: {self.model_name}')
        print(f'Classes : {NUM_CLASSES} fashion houses (2015-2019)')
        print(f'Brands  : {unique_brands}')
        print('─' * 65)
        print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | '
              f'{"Val Loss":>8} | {"Val Acc":>8} | {"LR":>10}')
        print('─' * 65)

        for epoch in range(1, epochs + 1):
            tr_loss, tr_acc = self.train_epoch(train_loader)
            vl_loss, vl_acc, _, _ = self.eval_epoch(val_loader)

            if isinstance(self.scheduler,
                          torch.optim.lr_scheduler.ReduceLROnPlateau):
                self.scheduler.step(vl_loss)
            else:
                self.scheduler.step()

            current_lr = self.optimizer.param_groups[0]['lr']

            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(vl_loss)
            self.history['train_acc'].append(tr_acc)
            self.history['val_acc'].append(vl_acc)

            print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2%} | '
                  f'{vl_loss:>8.4f} | {vl_acc:>8.2%} | {current_lr:>10.2e}')

            if early_stopping:
                early_stopping(vl_loss, self.model)
                if early_stopping.stop:
                    print(f'   Best val loss: {early_stopping.best_loss:.4f}')
                    break

        print('─' * 65)
        print(f'Training complete. Best Val Acc : {max(self.history["val_acc"]):.2%}')
        print(f'Model saved to    : best_model_2015_2019.pt')
        return self.history


def plot_training_history(history: dict, title: str = 'Training History — Vogue Runway 2015-2019'):
    """Plot loss and accuracy curves for 15-class fashion house classification."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    epochs = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], 'b-o', label='Train Loss', ms=4)
    ax1.plot(epochs, history['val_loss'],   'r-o', label='Val Loss',   ms=4)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Loss Curves'); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, history['train_acc'], 'b-o', label='Train Acc', ms=4)
    ax2.plot(epochs, history['val_acc'],   'r-o', label='Val Acc',   ms=4)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.set_title('Accuracy Curves'); ax2.legend(); ax2.grid(alpha=0.3)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

    plt.tight_layout()
    safe_name = title.lower().replace(' ', '_')
    plt.savefig(f'{safe_name}.png', dpi=150, bbox_inches='tight')
    plt.show()


print(f'Training engine defined for {NUM_CLASSES}-class dataset (2015-2019).')
print(f'Brands: {unique_brands}')

# transfer learning: resnet50 and efficientnet-b4
def build_resnet50(num_classes: int = NUM_CLASSES, mode: str = 'fine_tune',
                   dropout: float = 0.4):
    """
    ResNet50 for 15-class Vogue Runway 2015-2019 classification.
    Two modes:
    - 'feature_extract': freeze backbone, train only head
    - 'fine_tune'      : unfreeze all layers with smaller LR

    ResNet key innovation: Residual connections (skip connections)
    prevent vanishing gradients by providing identity shortcuts:
        output = F(x) + x
    This allows training of 50-152+ layer networks.
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    if mode == 'feature_extract':
        for param in model.parameters():
            param.requires_grad = False
        print('  ResNet50 backbone FROZEN (feature extraction mode)')
    else:
        # Freeze early layers — they hold generic features
        # Fine-tune deeper layers for fashion-specific features
        for name, param in model.named_parameters():
            if any(name.startswith(k) for k in ['layer1', 'layer2', 'conv1', 'bn1']):
                param.requires_grad = False
            else:
                param.requires_grad = True
        print('  ResNet50 TOP layers UNFROZEN (fine-tuning mode)')

    # Replace final classifier with 15-class head
    in_features = model.fc.in_features  # 2048 for ResNet50
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout * 0.5),
        nn.Linear(512, NUM_CLASSES)  # 15 fashion houses
    )
    return model


def build_efficientnet_b4(num_classes: int = NUM_CLASSES, mode: str = 'fine_tune',
                           dropout: float = 0.4):
    """
    EfficientNet-B4 for 15-class Vogue Runway 2015-2019 classification.

    EfficientNet key innovation: Compound scaling — simultaneously
    scale width, depth, and resolution with a fixed ratio.
    This achieves better accuracy with fewer parameters than
    architectures that scale only one dimension.

    B4 specs: 380px input, ~19M params, top-1 ImageNet: 83.0%
    """
    model = timm.create_model(
        'efficientnet_b4', pretrained=True,
        num_classes=0,     # remove classifier
        drop_rate=dropout
    )

    if mode == 'feature_extract':
        for param in model.parameters():
            param.requires_grad = False
        print('  EfficientNet-B4 backbone FROZEN')
    else:
        # Fine-tune: freeze early stages 0-2, train stages 3-6
        for name, param in model.named_parameters():
            if 'blocks.0' in name or 'blocks.1' in name:
                param.requires_grad = False
            else:
                param.requires_grad = True
        print('  EfficientNet-B4 TOP blocks UNFROZEN')

    in_features = model.num_features  # 1792 for B4
    classifier = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout * 0.5),
        nn.Linear(512, NUM_CLASSES)  # 15 fashion houses
    )
    model.classifier = classifier
    return model

In [ ]:
# ── Quick parameter summary ──────────────────────────────────
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


print(f'Model factory ready for {NUM_CLASSES}-class dataset (2015-2019).')
print(f'Brands: {unique_brands}')
print('\nBuilding models...')

resnet_fe  = build_resnet50(NUM_CLASSES, mode='feature_extract')
resnet_ft  = build_resnet50(NUM_CLASSES, mode='fine_tune')
effnet_ft  = build_efficientnet_b4(NUM_CLASSES, mode='fine_tune')

for name, mdl in [('ResNet50 (FE)', resnet_fe),
                   ('ResNet50 (FT)', resnet_ft),
                   ('EfficientNet-B4 (FT)', effnet_ft)]:
    total, trainable = count_params(mdl)
    print(f'\n  {name}: {total/1e6:.1f}M total | {trainable/1e6:.1f}M trainable')
    print(f'  Output classes: {NUM_CLASSES} ({", ".join(unique_brands)})')

# ── Constants for narrowed 2015-2019 dataset ──────────────────
LR           = 1e-3
WEIGHT_DECAY = 1e-4
NUM_EPOCHS   = 20   # more epochs since dataset is smaller

def get_optimizer_and_scheduler(model, lr=LR, epochs=NUM_EPOCHS,
                                  steps_per_epoch=None):
    """AdamW + Cosine Annealing LR schedule."""
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr * 0.01
    )
    return optimizer, scheduler


# Label smoothing slightly reduced — narrower class set is cleaner
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# ── Train 1: ResNet50 Feature Extraction ──────────────────────
print('\n' + '='*65)
print(f' MODEL 1: ResNet50 — FEATURE EXTRACTION ({NUM_CLASSES} brands, 2015-2019)')
print('='*65)
print(f' Brands: {unique_brands}')
print('='*65)

resnet_fe = resnet_fe.to(DEVICE)
opt_fe, sch_fe = get_optimizer_and_scheduler(resnet_fe, lr=LR)
es_fe = EarlyStopping(patience=7, path='resnet50_fe_best.pt')
# patience=7 (up from 5) since narrower data may need more epochs to converge

trainer_fe = Trainer(resnet_fe, opt_fe, sch_fe, criterion,
                     DEVICE, 'ResNet50-FeatureExtract')
hist_fe = trainer_fe.fit(train_loader, val_loader,
                          epochs=NUM_EPOCHS, early_stopping=es_fe)
plot_training_history(hist_fe, f'ResNet50 Feature Extraction — {NUM_CLASSES} Fashion Houses (2015-2019)')

# ── Train 2: ResNet50 Fine-Tuning (2015-2019, 15 brands) ──────
print('\n' + '='*65)
print(f' MODEL 2: ResNet50 — FINE-TUNING (MixUp | {NUM_CLASSES} brands, 2015-2019)')
print('='*65)
print(f' Brands: {unique_brands}')
print('='*65)

resnet_ft = resnet_ft.to(DEVICE)
opt_ft, sch_ft = get_optimizer_and_scheduler(
    resnet_ft, lr=3e-5, epochs=NUM_EPOCHS  # lower LR for narrower dataset
)
es_ft = EarlyStopping(patience=8, path='resnet50_ft_best.pt')
# patience=8: narrower class set may plateau slowly — give it more room

trainer_ft = Trainer(resnet_ft, opt_ft, sch_ft, criterion,
                     DEVICE, 'ResNet50-FineTune', use_mixup=True)
hist_ft = trainer_ft.fit(train_loader, val_loader,
                          epochs=NUM_EPOCHS, early_stopping=es_ft)
plot_training_history(hist_ft, f'ResNet50 Fine-Tuning — {NUM_CLASSES} Fashion Houses (2015-2019)')

# ── Diagnose Train/Val Accuracy Gap ───────────────────────────

# 1. Evaluate model on train set WITHOUT augmentation
train_eval_ds = VogueRunwayDataset(train_data, transform=val_transform)  # no augmentation
train_eval_loader = DataLoader(
    train_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)

model_to_eval = resnet_ft  # or whichever model you're checking
model_to_eval.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(train_eval_loader, desc='Evaluating train set'):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model_to_eval(imgs)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

train_acc_no_aug = accuracy_score(all_labels, all_preds)
print(f'\nTrain acc (no augmentation) : {train_acc_no_aug*100:.2f}%')
print(f'Val   acc                   : 98.99%')

# 2. Check class distribution in train vs val
print('\nClass distribution — Train vs Val:')
train_counter = Counter(train_data['label'])
val_counter   = Counter(val_data['label'])
print(f'{"Brand":<25} {"Train":>8} {"Val":>8}')
print('-' * 43)
for i, brand in enumerate(unique_brands):
    print(f'{brand:<25} {train_counter[i]:>8} {val_counter[i]:>8}')

# 3. Check if val set is leaking / too easy
print(f'\nTotal train samples : {len(train_data)}')
print(f'Total val samples   : {len(val_data)}')
print(f'Train/Val ratio     : {len(train_data)/len(val_data):.2f}x')

def full_evaluation(model, loader, class_names, model_name='Model',
                    checkpoint_path=None):
    """Compute all evaluation metrics and render confusion matrix."""
    if checkpoint_path and os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
        print(f'  📂 Loaded checkpoint: {checkpoint_path}')

    model.eval().to(DEVICE)
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='Evaluating'):
            imgs = imgs.to(DEVICE)
            with autocast():
                logits = model(imgs)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.tolist())
            all_probs.extend(probs.cpu().tolist())

    # ── Metrics ───────────────────────────────────────────────
    acc  = accuracy_score(all_labels, all_preds)
    f1_w = f1_score(all_labels, all_preds, average='weighted')
    f1_m = f1_score(all_labels, all_preds, average='macro')
    report = classification_report(all_labels, all_preds,
                                   target_names=class_names)

    print(f'\n📊 {model_name} — Evaluation Results (2015-2019 | {NUM_CLASSES} brands)')
    print('─' * 50)
    print(f'  Accuracy         : {acc:.4f} ({acc:.2%})')
    print(f'  F1 (Weighted)    : {f1_w:.4f}')
    print(f'  F1 (Macro)       : {f1_m:.4f}')
    print('\n  Per-Class Report:')
    print(report)

In [ ]:
 # ── Confusion Matrix ──────────────────────────────────────
    cm = confusion_matrix(all_labels, all_preds, normalize='true')
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_title(f'Normalised Confusion Matrix — {model_name}\n(2015-2019 | {NUM_CLASSES} Fashion Houses)',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name.lower().replace(" ","_")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    return {'acc': acc, 'f1_weighted': f1_w, 'f1_macro': f1_m,
            'preds': all_preds, 'labels': all_labels, 'probs': all_probs}


# ── Evaluate two models on narrowed 2015-2019 classes ─────────
# Using unique_brands (15 classes) instead of full label_names
results_fe = full_evaluation(resnet_fe, val_loader, unique_brands,
                              'ResNet50 FE', 'resnet50_fe_best.pt')
results_ft = full_evaluation(resnet_ft, val_loader, unique_brands,
                              'ResNet50 FT', 'resnet50_ft_best.pt')

# ── Model Comparison Table (2015-2019 | 15 brands) ────────────
comparison_data = {
    'Model': ['ResNet50 (FE)', 'ResNet50 (FT)'],
    'Mode': ['Feature Extract', 'Fine-Tune'],
    'Accuracy': [
        f"{results_fe['acc']:.2%}",
        f"{results_ft['acc']:.2%}",
    ],
    'F1 (Weighted)': [
        f"{results_fe['f1_weighted']:.4f}",
        f"{results_ft['f1_weighted']:.4f}",
    ],
}

df_compare = pd.DataFrame(comparison_data)
print(f'\n📋 Model Comparison — 2015-2019 | {NUM_CLASSES} Fashion Houses:')
print(df_compare.to_string(index=False))

# Bar chart comparison
models_with_results = ['ResNet50 (FE)', 'ResNet50 (FT)']
accs = [results_fe['acc'], results_ft['acc']]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#4ECDC4', '#FF6B6B']
bars = ax.bar(models_with_results, accs, color=colors, edgecolor='black', width=0.4)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{acc:.2%}', ha='center', fontweight='bold', fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title(f'Model Performance Comparison\n2015-2019 | {NUM_CLASSES} Fashion Houses',
             fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison_2015_2019.png', dpi=150, bbox_inches='tight')
plt.show()

import cv2

class GradCAM:
    """Grad-CAM implementation (no external dependencies)."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def generate(self, img_tensor: torch.Tensor,
                 target_class: int = None):
        """Generate CAM heatmap for target_class (or predicted class)."""
        self.model.eval()
        img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
        img_tensor.requires_grad = True

        # Forward
        logits = self.model(img_tensor)
        if target_class is None:
            target_class = logits.argmax(dim=1).item()

        # Backward for target class
        self.model.zero_grad()
        one_hot = torch.zeros_like(logits)
        one_hot[0, target_class] = 1.0
        logits.backward(gradient=one_hot, retain_graph=True)

        # Grad-CAM computation
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE),
                            mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        # Normalize to [0, 1]
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam, target_class


def overlay_cam(img_tensor: torch.Tensor, cam: np.ndarray,
                alpha: float = 0.5):
    """Overlay CAM heatmap on original image."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img  = (img_tensor.cpu() * std + mean).clamp(0, 1)
    img  = img.permute(1, 2, 0).numpy()

    heatmap = cv2.applyColorMap(
        (cam * 255).astype(np.uint8), cv2.COLORMAP_JET
    )
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    overlay = alpha * heatmap + (1 - alpha) * img
    return np.clip(overlay, 0, 1)


# ── Instantiate Grad-CAM on ResNet50 FT (best model) ──────────
# Target: last convolutional layer in layer4
target_layer = resnet_ft.layer4[-1].conv3  # last conv in final ResNet block
gradcam = GradCAM(resnet_ft, target_layer)
print(f'✅ Grad-CAM ready on ResNet50 FT — {NUM_CLASSES} fashion houses (2015-2019)')

# ── Generate Grad-CAM for samples (ResNet50 FT | 2015-2019) ───
n_cam_samples = 5
fig, axes = plt.subplots(n_cam_samples, 3,
                          figsize=(12, n_cam_samples * 4))
fig.suptitle(f'🔍 Grad-CAM: ResNet50 FT — {NUM_CLASSES} Fashion Houses (2015-2019)',
             fontsize=16, fontweight='bold')

val_iter = iter(val_loader)
imgs_batch, labels_batch = next(val_iter)

for row_idx in range(n_cam_samples):
    img_t  = imgs_batch[row_idx]
    true_l = labels_batch[row_idx].item()

    cam, pred_class = gradcam.generate(img_t, target_class=None)
    overlay = overlay_cam(img_t, cam)

    # Denorm original
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    orig = (img_t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    axes[row_idx][0].imshow(orig)
    axes[row_idx][0].set_title(f'Original\nTrue: {unique_brands[true_l]}',
                                fontsize=9)
    axes[row_idx][0].axis('off')

    axes[row_idx][1].imshow(cam, cmap='jet')
    axes[row_idx][1].set_title('Grad-CAM Heat', fontsize=9)
    axes[row_idx][1].axis('off')

    color = 'green' if pred_class == true_l else 'red'
    axes[row_idx][2].imshow(overlay)
    axes[row_idx][2].set_title(
        f'Overlay\nPred: {unique_brands[pred_class]}',
        fontsize=9, color=color
    )
    axes[row_idx][2].axis('off')

plt.tight_layout()
plt.savefig('gradcam_resnet50_ft_2015_2019.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 Grad-CAM Interpretation (2015-2019 | {NUM_CLASSES} brands):')
print('  Hot regions (red/yellow) = what the model uses for prediction')
print('  Good: activates on garment texture/silhouette')
print('  Bad: activates on background/model face (spurious correlation)')

def error_analysis(model, loader, class_names, model_name='Model',
                   n_show=20):
    """Find and visualize misclassified samples."""
    model.eval().to(DEVICE)
    errors = []  # (img_tensor, true_label, pred_label, confidence)

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            with autocast():
                logits = model(imgs)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)

            for i in range(len(labels)):
                if preds[i].item() != labels[i].item():
                    errors.append((
                        imgs[i].cpu(),
                        labels[i].item(),
                        preds[i].item(),
                        probs[i][preds[i]].item()
                    ))

    print(f'\n{model_name}: {len(errors)} misclassifications '
          f'({len(errors)/len(loader.dataset):.1%} error rate)')

    # Analyse which pairs are most confused
    confusion_pairs = Counter()
    for _, true_l, pred_l, _ in errors:
        confusion_pairs[(class_names[true_l], class_names[pred_l])] += 1
    print('\n  Top 10 confused pairs (True → Predicted):')
    for (true, pred), count in confusion_pairs.most_common(10):
        print(f'    {true:25s} → {pred:25s}  [{count} times]')

    # Show misclassified images
    n_show = min(n_show, len(errors))
    cols   = 5
    rows   = math.ceil(n_show / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()
    fig.suptitle(f'Misclassified Samples — {model_name}\n(2015-2019 | {NUM_CLASSES} Fashion Houses)',
                 fontsize=14, fontweight='bold')

    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    for i, (img_t, true_l, pred_l, conf) in enumerate(errors[:n_show]):
        img = (img_t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[i].imshow(img)
        axes[i].set_title(
            f'True: {class_names[true_l]}\n'
            f'Pred: {class_names[pred_l]} ({conf:.0%})',
            fontsize=8, color='red'
        )
        axes[i].axis('off')

    for i in range(n_show, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig(f'error_analysis_{model_name.lower().replace(" ", "_")}_2015_2019.png',
                dpi=120, bbox_inches='tight')
    plt.show()

    return errors, confusion_pairs


# ── Error Analysis on ResNet50 FT (2015-2019 | 15 brands) ─────
resnet_ft.load_state_dict(
    torch.load('resnet50_ft_best.pt', map_location=DEVICE)
)
errors_ft, confusion_pairs_ft = error_analysis(
    resnet_ft, val_loader, unique_brands, 'ResNet50 FT'
)

# ── Error Analysis on ResNet50 FE (2015-2019 | 15 brands) ─────
resnet_fe.load_state_dict(
    torch.load('resnet50_fe_best.pt', map_location=DEVICE)
)
errors_fe, confusion_pairs_fe = error_analysis(
    resnet_fe, val_loader, unique_brands, 'ResNet50 FE'
)

print(f"""
─────────────────────────────────────────────────────────
🔬 Error Analysis — 2015-2019 | {NUM_CLASSES} Fashion Houses
─────────────────────────────────────────────────────────

Common confusion patterns and WHY they occur:

1. Balenciaga ↔ Saint Laurent
   → Both shifted toward dark, structured silhouettes during
     2015-2019. The CNN's texture detectors struggle to separate
     these since the key discriminator is cut/proportion, not
     surface pattern.

2. Gucci ↔ Versace
   → Both use maximalist print language during this period.
     High visual frequency textures dominate early conv layers.
     Logo-level features (critical discriminators) may be
     small or occluded.

3. Armani Privé ↔ Chanel
   → Both use refined, understated elegance with similar
     colour palettes (nude, black, ivory). Without logo
     cues the CNN relies on texture alone — insufficient
     for this pair.

Solution: Attention mechanisms (ViT, CBAM) that focus on
brand-critical regions regardless of scale.
─────────────────────────────────────────────────────────
""")

# ── Optuna HPO — 2015-2019 | 15 brands | FT + FE ──────────────────────────
import optuna
from optuna.trial import TrialState
optuna.logging.set_verbosity(optuna.logging.WARNING)

OPTUNA_EPOCHS = 5
OPTUNA_TRIALS = 20

# ── Shared objective factory ──────────────────────────────────────────────
def make_objective(mode: str):
    """Returns an Optuna objective for 'fine_tune' or 'feature_extract'."""
    def objective(trial: optuna.Trial) -> float:
        lr_           = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
        dropout_      = trial.suggest_float('dropout', 0.2, 0.6, step=0.1)
        weight_decay_ = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
        label_smooth_ = trial.suggest_float('label_smoothing', 0.05, 0.2, step=0.05)

        model = build_resnet50(
            NUM_CLASSES,           # 15 brands (2015-2019, CK merged)
            mode=mode,
            dropout=dropout_
        ).to(DEVICE)

        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr_, weight_decay=weight_decay_
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=OPTUNA_EPOCHS
        )
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth_)
        scaler    = GradScaler()   # created ONCE per trial, not per batch

        best_val_acc = 0.0
        for epoch in range(OPTUNA_EPOCHS):
            # ── Train ──────────────────────────────────────────────────
            model.train()
            for imgs, labels in train_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                with autocast():
                    loss = criterion(model(imgs), labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            scheduler.step()

In [ ]:
    # ── Validate ───────────────────────────────────────────────
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for imgs, labels in val_loader:
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    with autocast():
                        preds = model(imgs).argmax(dim=1)
                    correct += (preds == labels).sum().item()
                    total   += labels.size(0)

            val_acc      = correct / total
            best_val_acc = max(best_val_acc, val_acc)

            trial.report(val_acc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return best_val_acc
    return objective


# ── Run search for both models ────────────────────────────────────────────
best_params = {}

for mode, label in [('fine_tune', 'ResNet50 FT'), ('feature_extract', 'ResNet50 FE')]:
    print(f'\n Optuna search — {label} '
          f'({OPTUNA_TRIALS} trials × {OPTUNA_EPOCHS} epochs) '
          f'| {NUM_CLASSES} brands (2015–2019)')

    study = optuna.create_study(
        study_name=f'resnet50_{mode}_2015_2019',
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
    )
    study.optimize(
        make_objective(mode),
        n_trials=OPTUNA_TRIALS,
        show_progress_bar=True
    )

    best = study.best_trial
    best_params[label] = best.params
    print(f'\n {label} — Best val_acc: {best.value:.4f}')
    print(f'   Brands : {NUM_CLASSES} ({", ".join(unique_brands[:4])}, ...)')
    print(f'   Years  : 2015–2019')
    print('   Params :')
    for k, v in best.params.items():
        print(f'     {k}: {v}')


# ── Summary ───────────────────────────────────────────────────────────────
print('\n' + '─'*55)
print(f'HPO Summary — 2015-2019 | {NUM_CLASSES} Fashion Houses')
print('─'*55)
for model_label, params in best_params.items():
    print(f'\n  {model_label}')
    for k, v in params.items():
        print(f'    {k}: {v}')

# ── Visualise Optuna results — ResNet50 FT & FE (2015-2019 | 15 brands) ──
for mode, label, stdy in [
    ('fine_tune',       'ResNet50 FT', study_ft),
    ('feature_extract', 'ResNet50 FE', study_fe),
]:
    fig = optuna.visualization.matplotlib.plot_param_importances(stdy)
    plt.title(f'Hyperparameter Importance — {label}\n(2015-2019 | {NUM_CLASSES} brands)',
              fontsize=13)
    plt.tight_layout()
    plt.savefig(f'optuna_param_importance_{mode}_2015_2019.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    fig2 = optuna.visualization.matplotlib.plot_optimization_history(stdy)
    plt.title(f'Optimisation History — {label}\n(2015-2019 | {NUM_CLASSES} brands)',
              fontsize=13)
    plt.tight_layout()
    plt.savefig(f'optuna_history_{mode}_2015_2019.png',
                dpi=150, bbox_inches='tight')
    plt.show()

class EnsembleModel(nn.Module):
    """
    Soft-voting ensemble of ResNet50 FT + ResNet50 FE.

    Theory: Ensembling reduces variance without increasing bias.
    FT and FE have different gradient flows during training —
    FT updates all layers (low bias, higher variance) while FE
    updates only the head (higher bias, lower variance). Their
    errors are partially independent → averaging cancels them.

    Expected gain: +1-2% accuracy over the single best model.
    """
    def __init__(self, models: list, weights: list = None):
        super().__init__()
        self.models  = nn.ModuleList(models)
        self.weights = weights or [1.0 / len(models)] * len(models)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        probs = torch.zeros(x.size(0), NUM_CLASSES, device=x.device)
        for model, w in zip(self.models, self.weights):
            model.eval()
            with autocast():
                logits = model(x)
            probs = probs + w * F.softmax(logits, dim=1)
        return probs   # probabilities, not logits


# ── Load best weights ─────────────────────────────────────────
resnet_ft.load_state_dict(
    torch.load('resnet50_ft_best.pt', map_location=DEVICE))
resnet_fe.load_state_dict(
    torch.load('resnet50_fe_best.pt', map_location=DEVICE))

# FT gets higher weight — it fine-tunes all layers and typically
# achieves higher accuracy on fashion texture/detail tasks.
ensemble = EnsembleModel(
    models=[resnet_ft.to(DEVICE), resnet_fe.to(DEVICE)],
    weights=[0.65, 0.35]
).to(DEVICE)

# ── Evaluate Ensemble ─────────────────────────────────────────
ensemble.eval()
correct = total = 0
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc='Ensemble eval'):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        probs  = ensemble(imgs)
        preds  = probs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

ens_acc = correct / total
ens_f1  = f1_score(all_labels, all_preds, average='weighted')

# Compare against the stronger single model (FT)
best_single_acc = results_ft['acc']   # set from your earlier eval block

print(f'\n Ensemble (FT 65% + FE 35%) — 2015-2019 | {NUM_CLASSES} brands')
print(f'   Accuracy        : {ens_acc:.4f} ({ens_acc:.2%})')
print(f'   F1 (weighted)   : {ens_f1:.4f}')
print(f'   vs ResNet50 FT  : {best_single_acc:.2%}')
print(f'   Ensemble gain   : +{(ens_acc - best_single_acc):.2%}')

def extract_embeddings(model, loader):
    """
    Extract penultimate-layer embeddings via a forward hook.

    For ResNet50 the penultimate layer is the global average pool
    output — a (B, 2048) vector per image before the classifier head.

    t-SNE projects these 2048-D vectors to 2D while preserving
    local neighbourhood structure. Tight, separated clusters indicate
    the model has learned discriminative brand representations.
    """
    embeddings, labels_out = [], []
    hook_output = []

    def hook_fn(module, inp, out):
        hook_output.append(out.detach().cpu())

    # ResNet50: hook on the adaptive average pool (2048-D output)
    handle = model.avgpool.register_forward_hook(hook_fn)

    model.eval()
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc='Extracting features'):
            imgs = imgs.to(DEVICE)
            hook_output.clear()
            with autocast():
                _ = model(imgs)
            feat = hook_output[0].squeeze(-1).squeeze(-1)  # [B, 2048]
            embeddings.append(feat.numpy())
            labels_out.extend(lbls.tolist())

    handle.remove()
    embeddings = np.concatenate(embeddings, axis=0)
    return embeddings, np.array(labels_out)


# ── Extract from ResNet50 FT (stronger model) ─────────────────
print(' Extracting ResNet50 FT embeddings (2015-2019 | 15 brands)...')
small_val_loader = DataLoader(
    val_ds, batch_size=64, shuffle=True,
    num_workers=NUM_WORKERS
)

feats, feat_labels = extract_embeddings(resnet_ft, small_val_loader)
print(f'  Feature matrix shape: {feats.shape}')   # (N, 2048)

# ── t-SNE ────────────────────────────────────────────────────
print(' Running t-SNE (perplexity=30)...')
tsne = TSNE(
    n_components=2, perplexity=30, random_state=SEED,
    n_iter=1000, learning_rate='auto', init='pca'
)
tsne_coords = tsne.fit_transform(feats)

In [ ]:
# ── Plot ──────────────────────────────────────────────────────
palette = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))
fig, ax  = plt.subplots(figsize=(14, 12))

for class_id, brand in enumerate(unique_brands):
    mask = feat_labels == class_id
    ax.scatter(
        tsne_coords[mask, 0], tsne_coords[mask, 1],
        c=[palette[class_id]], label=brand,
        alpha=0.7, s=20, edgecolors='none'
    )

ax.legend(loc='best', fontsize=8, ncol=2, framealpha=0.9)
ax.set_title(
    f't-SNE — ResNet50 FT Embedding Space\n'
    f'2015-2019 | {NUM_CLASSES} Fashion Houses | {feats.shape[0]} images',
    fontsize=14, fontweight='bold'
)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('tsne_resnet50_ft_2015_2019.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n Tight clusters = discriminative brand representations.')
print('   Overlapping clusters = visually similar brands (expected: Bal↔SL, Gucci↔Versace).')

import gradio as gr
from PIL import Image

# ── Load best model for deployment ───────────────────────────
deploy_model = build_resnet50(
    NUM_CLASSES,          # 15 brands (2015-2019, CK merged)
    mode='fine_tune',
    dropout=0.0           # dropout off at inference
).to(DEVICE)
deploy_model.load_state_dict(
    torch.load('resnet50_ft_best.pt', map_location=DEVICE))
deploy_model.eval()
print(f'ResNet50 FT loaded — {NUM_CLASSES} brands (2015-2019).')

# Inference transform (no augmentation)
inference_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


def predict_brand(pil_image: Image.Image):
    """
    Inference function for Gradio.
    Returns top-5 brand predictions with confidence scores.
    Labels drawn from unique_brands (15 classes, 2015-2019).
    """
    if pil_image is None:
        return {}

    img_t = inference_transform(pil_image.convert('RGB'))
    img_t = img_t.unsqueeze(0).to(DEVICE)

    with torch.no_grad(), autocast():
        logits = deploy_model(img_t)
        probs  = F.softmax(logits, dim=1)[0]

    top5_probs, top5_ids = torch.topk(probs, k=5)
    result = {
        unique_brands[idx.item()]: round(prob.item(), 4)
        for prob, idx in zip(top5_probs, top5_ids)
    }
    return result

In [ ]:
# ── Gradio Interface ──────────────────────────────────────────
brand_list = ', '.join(unique_brands)

demo = gr.Interface(
    fn=predict_brand,
    inputs=gr.Image(
        type='pil',
        label='Upload Runway Image'
    ),
    outputs=gr.Label(
        num_top_classes=5,
        label='Fashion House Prediction (Top 5)'
    ),
    title='VOGUE RUNWAY CLASSIFIER — 2015-2019',
    description=(
        f'Upload a runway fashion image to identify the fashion house.\n\n'
        f'**Model:** ResNet50 fine-tuned on Vogue Runway (2015-2019)\n'
        f'**Classes ({NUM_CLASSES}):** {brand_list}\n\n'
        f'Calvin Klein Collection is merged into Calvin Klein.'
    ),
    examples=[],
    theme=gr.themes.Soft(),
    allow_flagging='never'
)

demo.launch(share=True, debug=False)
print('Gradio app launched!')